---
last_verified: 2026-09-29
tool_version: n/a
---

# Comparing Semgrep rule designs — search, context-constrained, and taint

**Purpose:** Four rules written for the same vulnerability class (command injection through
a shell sink in Python) are scored against the same labelled corpus, so the effect of the
*rule design* — not the rule's subject — is visible as a precision/recall table.

Every design in this notebook is expressed twice: as a Semgrep rule in the tool's own YAML
syntax, and as an executable Python stand-in that implements the same matching semantics
over the same fixture source. The Python stand-ins let the comparison run in any
environment; where the `semgrep` binary is present, Step 7 runs the real rules and the two
result sets can be compared.

## When to use

Use this comparison when the choice is between **rule designs**, not between tools:

- Picking a pattern vs taint rule for a new finding class, with a measured cost for each.
- Deciding how aggressively to narrow an over-firing search rule.
- Auditing an existing rule set by grouping findings by rule design and comparing the
  false-positive rate of each group.
- Deciding whether an existing rule is worth rewriting as a taint rule, or whether the
  cheaper fix is to widen its source list.

Use the design comparison in `semgrep/docs/comparing-rule-writing-approaches.md` instead
when the question is which *operator* to reach for inside one rule.

## Prerequisites

- Python 3 (standard library only — `ast`, `json`, `shutil`, `tempfile`).
- Optional: the `semgrep` CLI on `PATH`, for Step 7. The comparison itself does not
  require it; Step 7 reports and skips cleanly when the binary is absent.

## Step 1 — The labelled fixture corpus

Eight short Python fragments, each with a ground-truth label. Three are genuinely
injectable (user input reaches a shell sink); five are not, for three distinct reasons:
the command is a constant, the interpolated value is a program constant, or the
interpolated value reaches the process through an argument vector rather than a shell.

A corpus built only from vulnerable snippets measures recall and hides everything that
matters for precision, so the negative cases carry as much weight as the positive ones.

In [ ]:
FIXTURES = [
    # (id, source, expected_injectable)
    ("const_os_system", 'os.system("ls -la /tmp")', False),
    ("concat_literal", 'name = "report"\nos.system("ls -la " + name)', False),
    ("concat_tainted", 'name = request.args.get("name")\nos.system("ls -la " + name)', True),
    ("fstring_literal", 'who = "all"\nos.system(f"ls {who}")', False),
    ("fstring_tainted", 'who = request.args.get("who")\nos.system(f"ls {who}")', True),
    ("shell_true_literal", 'subprocess.run("uptime", shell=True)', False),
    ("shell_true_tainted", 'cmd = request.args.get("c")\nsubprocess.run(f"sh -c {cmd}", shell=True)', True),
    ("list_args_tainted", 'p = request.args.get("p")\nsubprocess.run(["ls", p], shell=False)', False),
]

for fid, src, unsafe in FIXTURES:
    print(f"{'UNSAFE' if unsafe else 'safe  '}  {fid}")

## Step 2 — Shell-sink extraction

Each design below needs the same two primitives: *which calls are shell sinks*, and *which
values in the source are untrusted*. Both are small enough to express directly on the
abstract syntax tree, which is what makes the designs comparable rather than merely
describable.

In [ ]:
import ast


def dotted(node):
    """('os', 'system') for os.system, ('subprocess', 'run') for subprocess.run."""
    parts = []
    while isinstance(node, ast.Attribute):
        parts.append(node.attr)
        node = node.value
    if isinstance(node, ast.Name):
        parts.append(node.id)
    return tuple(reversed(parts))


SHELL_RUNNERS = {
    ("subprocess", "run"),
    ("subprocess", "call"),
    ("subprocess", "Popen"),
    ("subprocess", "check_output"),
}

# Full source list: the framework entry points a web handler actually uses.
FULL_SOURCES = {
    ("request", "args", "get"),
    ("request", "values", "get"),
    ("request", "form", "get"),
    ("request", "json"),
    ("flask", "request", "args", "get"),
    ("os", "environ", "get"),
}

# Narrow source list: a plausible first draft that covers only two of the six.
NARROW_SOURCES = {("os", "environ", "get"), ("sys", "argv")}


def is_source_call(node, sources):
    return isinstance(node, ast.Call) and dotted(node.func) in sources


def sinks(tree):
    """Yield (call, first-argument) for every call that hands a string to a shell."""
    for node in ast.walk(tree):
        if not isinstance(node, ast.Call) or not node.args:
            continue
        callee = dotted(node.func)
        if callee == ("os", "system"):
            yield node, node.args[0]
        elif callee in SHELL_RUNNERS:
            shell = next((kw.value for kw in node.keywords if kw.arg == "shell"), None)
            if isinstance(shell, ast.Constant) and shell.value is True:
                yield node, node.args[0]


def tainted_names(tree, sources):
    """Names bound directly to a value returned by one of `sources`."""
    names = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign) and any(
            is_source_call(n, sources) for n in ast.walk(node.value)
        ):
            names.update(t.id for t in node.targets if isinstance(t, ast.Name))
    return names


def arg_is_tainted(arg, names, sources):
    if is_source_call(arg, sources):
        return True
    return any(isinstance(n, ast.Name) and n.id in names for n in ast.walk(arg))


def is_literal_string(node):
    return isinstance(node, ast.Constant) and isinstance(node.value, str)


parsed = {fid: ast.parse(src) for fid, src, _ in FIXTURES}
print({fid: len(list(sinks(tree))) for fid, tree in parsed.items()})

## Step 3 — The four rule designs in Semgrep syntax

- **A — broad search rule.** Matches the sink itself. Maximum recall, no notion of trust.
- **B — context-constrained search rule.** Same sinks, narrowed with `pattern-not` so a
  constant command string does not match.
- **C — taint rule, complete source list.** Tracks untrusted values from the framework
  entry points to the sink.
- **D — taint rule, narrow source list.** Same engine as C, two sources. D exists to
  isolate one variable: the engine is identical, the source list is not.

In [ ]:
SEMGREP_RULES = {
    "A": """
rules:
  - id: search-any-shell-sink
    languages: [python]
    severity: WARNING
    message: A process is started through a shell sink. Review how the command string is built.
    patterns:
      - pattern-either:
          - pattern: os.system($CMD)
          - pattern: subprocess.run($CMD, shell=True, ...)
          - pattern: subprocess.Popen($CMD, shell=True, ...)
          - pattern: subprocess.call($CMD, shell=True, ...)
          - pattern: subprocess.check_output($CMD, shell=True, ...)
""",
    "B": """
rules:
  - id: search-interpolated-shell-sink
    languages: [python]
    severity: WARNING
    message: The shell command is built from an expression rather than a constant.
    patterns:
      - pattern-either:
          - pattern: os.system($CMD)
          - pattern: subprocess.run($CMD, shell=True, ...)
      - pattern-not: os.system("...")
      - pattern-not: subprocess.run("...", shell=True, ...)
""",
    "C": """
rules:
  - id: taint-request-to-shell
    languages: [python]
    severity: ERROR
    mode: taint
    message: Untrusted input reaches a shell sink.
    pattern-sources:
      - patterns:
          - pattern-either:
              - pattern: request.args.get(...)
              - pattern: request.values.get(...)
              - pattern: request.form.get(...)
              - pattern: request.json()
              - pattern: flask.request.args.get(...)
              - pattern: os.environ.get(...)
    pattern-sanitizers:
      - pattern: shlex.quote(...)
    pattern-sinks:
      - patterns:
          - pattern-either:
              - pattern: os.system($CMD)
              - pattern: subprocess.run($CMD, shell=True, ...)
""",
    "D": """
rules:
  - id: taint-narrow-sources-to-shell
    languages: [python]
    severity: ERROR
    mode: taint
    message: Environment or argv input reaches a shell sink.
    pattern-sources:
      - patterns:
          - pattern-either:
              - pattern: os.environ.get(...)
              - pattern: sys.argv
    pattern-sinks:
      - patterns:
          - pattern-either:
              - pattern: os.system($CMD)
              - pattern: subprocess.run($CMD, shell=True, ...)
""",
}

for design, rule in SEMGREP_RULES.items():
    print(f"--- design {design} ---")
    print(rule.strip())
    print()

## Step 4 — Executing each design

Each rule above is mirrored by a detector with the same matching semantics: same sinks,
same exclusions, same source list. Running all four over the corpus turns "this design
is less precise" into a number.

In [ ]:
def design_a():
    """Any shell sink matches."""
    return [fid for fid, t in parsed.items() if any(True for _ in sinks(t))]


def design_b():
    """Shell sinks whose command argument is not a bare string constant."""
    out = []
    for fid, t in parsed.items():
        if any(not is_literal_string(arg) for _, arg in sinks(t)):
            out.append(fid)
    return out


def design_c():
    """Taint tracking with the full source list."""
    out = []
    for fid, t in parsed.items():
        names = tainted_names(t, FULL_SOURCES)
        if any(arg_is_tainted(arg, names, FULL_SOURCES) for _, arg in sinks(t)):
            out.append(fid)
    return out


def design_d():
    """Same taint engine, two-entry source list."""
    out = []
    for fid, t in parsed.items():
        names = tainted_names(t, NARROW_SOURCES)
        if any(arg_is_tainted(arg, names, NARROW_SOURCES) for _, arg in sinks(t)):
            out.append(fid)
    return out


DESIGNS = {"A": design_a, "B": design_b, "C": design_c, "D": design_d}

for name, fn in DESIGNS.items():
    print(f"{name}: {sorted(fn())}")

## Step 5 — Scoring against the labels

Each design is scored on the same corpus. Three of the four reach full recall, so the
recall column cannot separate them — the precision column is what exposes what each design
costs.

In [ ]:
def score(design_fn):
    tp = fp = fn = tn = 0
    for fid, _, unsafe in FIXTURES:
        flagged = fid in design_fn()
        if flagged and unsafe:
            tp += 1
        elif flagged:
            fp += 1
        elif unsafe:
            fn += 1
        else:
            tn += 1
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn, "precision": precision, "recall": recall, "f1": f1}


def row(label, m):
    return (f"{label:<34}{m['tp']:>4}{m['fp']:>4}{m['fn']:>4}{m['tn']:>4}"
            f"{m['precision']:>11.2f}{m['recall']:>9.2f}{m['f1']:>8.2f}")


header = f"{'design':<34}{'TP':>4}{'FP':>4}{'FN':>4}{'TN':>4}{'precision':>11}{'recall':>9}{'F1':>8}"
print(header)
print("-" * len(header))
for name, fn in DESIGNS.items():
    print(row(f"{name} — {fn.__doc__.splitlines()[0][:28]}", score(fn)))

## Step 6 — Reading the matrix

The interesting comparison is C against D: the same engine, the same sinks, the same
corpus, and a recall of 1.0 against 0.0. Rule *coverage* is a property of the source list;
rule *power* is a property of the engine. Confusing the two is the most common way a
working rule silently stops finding anything when the application is refactored.

In [ ]:
results = {name: score(fn) for name, fn in DESIGNS.items()}
ranked = sorted(results.items(), key=lambda kv: (-kv[1]["f1"], kv[0]))

print("ranked by F1 (ties broken by design id):")
for i, (name, m) in enumerate(ranked, 1):
    print(f"  {i}. {name}  precision={m['precision']:.2f}  recall={m['recall']:.2f}")

over_firing = {n: m for n, m in results.items() if m["fp"] > 0}
silent = {n: m for n, m in results.items() if m["fn"] > 0}
print()
print(f"designs that over-fire : {sorted(over_firing)}")
print(f"designs that under-fire: {sorted(silent)}")

cost = {"A": "cheapest to run, no data-flow reasoning", "B": "cheapest to run, narrower match set",
        "C": "data-flow reasoning per file, slowest of the four", "D": "same cost as C, narrower sources"}
print()
for name in sorted(results):
    print(f"  {name}: {cost[name]}")

## Step 7 — Running the same rules with the CLI

The four rule files are written out and handed to the real binary where it is available.
Without the binary the step reports that and stops — the comparison above is unaffected,
because the detectors and the rules encode the same semantics.

In [ ]:
import json
import shutil
import subprocess
import tempfile
from pathlib import Path

SEMGREP_BIN = shutil.which("semgrep")
WORK = Path(tempfile.mkdtemp(prefix="semgrep-rule-comparison-"))
for name, rule in SEMGREP_RULES.items():
    (WORK / f"design_{name}.yaml").write_text(rule.strip() + "\n", encoding="utf-8")
for fid, src, _ in FIXTURES:
    (WORK / f"{fid}.py").write_text(src + "\n", encoding="utf-8")

print(f"fixtures and rule files written to {WORK}")

if SEMGREP_BIN is None:
    print("semgrep is not on PATH — skipping the CLI run; the detectors above already scored every design.")
else:
    for name in sorted(SEMGREP_RULES):
        proc = subprocess.run(
            [SEMGREP_BIN, "scan", "--config", str(WORK / f"design_{name}.yaml"),
             "--json", "--quiet", "--disable-version-check", str(WORK)],
            capture_output=True, text=True,
        )
        if proc.returncode not in (0, 1):
            print(f"design {name}: scan failed (exit {proc.returncode})\n{proc.stderr[:400]}")
            continue
        payload = json.loads(proc.stdout)
        hits = sorted({f["path"].rsplit("/", 1)[-1][:-3] for f in payload.get("results", [])})
        print(f"design {name}: {hits}")

## Verify

The four expected scores are asserted so the comparison fails loudly if a fixture, a
detector, or a source list is edited without the assertions being revisited.

In [ ]:
EXPECTED = {
    "A": {"tp": 3, "fp": 4, "fn": 0, "tn": 1, "precision": 3 / 7, "recall": 1.0},
    "B": {"tp": 3, "fp": 2, "fn": 0, "tn": 3, "precision": 0.6, "recall": 1.0},
    "C": {"tp": 3, "fp": 0, "fn": 0, "tn": 5, "precision": 1.0, "recall": 1.0},
    "D": {"tp": 0, "fp": 0, "fn": 3, "tn": 5, "precision": 0.0, "recall": 0.0},
}

for name, want in EXPECTED.items():
    got = results[name]
    for key in ("tp", "fp", "fn", "tn"):
        assert got[key] == want[key], f"design {name}: {key}={got[key]}, expected {want[key]}"
    for key in ("precision", "recall"):
        assert abs(got[key] - want[key]) < 1e-9, f"design {name}: {key}={got[key]}, expected {want[key]}"

assert ranked[0][0] == "C", "the taint rule with a complete source list should rank first"
assert sorted(over_firing) == ["A", "B"], sorted(over_firing)
assert sorted(silent) == ["D"], sorted(silent)
assert results["C"]["fn"] == 0 and results["D"]["tp"] == 0
print("all comparison checks passed")

## Common errors

| Error | Cause | Fix |
|---|---|---|
| A taint rule reports nothing after a refactor | The source list names framework entry points the application no longer uses | Re-derive the source list from the current handler code; design D above is the same failure, frozen |
| A `pattern-not` clause silently excludes a real finding | The exclusion pattern is broader than intended and matches the tainted case too | Exclude the narrow case (`os.system("...")`) rather than the whole sink, and re-run the comparison |
| A search rule is disabled repo-wide after triage | It matches every shell sink, so the finding count is unmanageable | Narrow to the interpolated form (design B) or move to taint (design C) before triaging |
| The argument-vector form is reported as injectable | The rule matches `subprocess.*` without checking `shell` | Match the `shell=True` keyword; the `list_args_tainted` fixture exists to catch this |
| Precision looks fine on a positives-only corpus | With no negative fixtures every rule scores 1.0 | Keep the negative fixtures and re-score before changing a rule |

## Choosing a design

- **Reach for a broad search rule first.** It is the cheapest to run and the easiest to
  review; the number of false positives it produces is a measurement, not a defect.
- **Narrow with `pattern-not` before escalating to taint.** Design B keeps design A's
  recall and removes half the noise for the cost of one clause.
- **Move to taint when interpolation alone cannot tell safe from unsafe.** Design B cannot
  separate `concat_literal` from `concat_tainted` — they are syntactically identical.
- **Review the source list, not the engine, when a taint rule goes quiet.** Designs C and
  D differ only in that list, and only one of them finds anything.

## References

- `semgrep/docs/semgrep-rule-writing-reference.md` — the rule keys and operators used by
  the four designs above.
- `semgrep/docs/comparing-rule-writing-approaches.md` — operator-level choices inside a
  single rule, which this notebook deliberately holds fixed.
- `semgrep/docs/semgrep-rule-performance-optimization.md` — the cost side of the
  search-vs-taint tradeoff summarised in Step 6.
- `semgrep/manifests/diff-aware-semgrep-ci.yaml` — how rule output is consumed in CI, once
  the noisy designs have been narrowed.